<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Comprehensive Distributed Training Guide</h1>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This notebook provides a comprehensive guide to distributed training using LLaMA-Factory, covering:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>DeepSpeed Training</strong>: ZeRO stages and memory optimization</li>
<li style="margin:6px 0;"><strong>FSDP Training</strong>: Fully Sharded Data Parallel</li>
<li style="margin:6px 0;"><strong>Ray Training</strong>: Distributed computing framework</li>
<li style="margin:6px 0;"><strong>Multi-Node Setup</strong>: Scaling across multiple machines</li>
<li style="margin:6px 0;"><strong>Performance Optimization</strong>: Communication and memory efficiency</li>
<li style="margin:6px 0;"><strong>Monitoring and Debugging</strong>: Distributed training analysis</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#setup-and-installation">Setup and Installation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#deepspeed-training">DeepSpeed Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#fsdp-training">FSDP Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#ray-training">Ray Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#multi-node-setup">Multi-Node Setup</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#performance-optimization">Performance Optimization</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#monitoring-and-debugging">Monitoring and Debugging</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#best-practices">Best Practices</a></li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Setup and Installation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">First, let's install the required dependencies for distributed training.</p>
</div>


In [ ]:
# Install distributed training dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets accelerate deepspeed
%pip install ray[tune] ray[train] ray[serve]
%pip install torch-fidelity  # For distributed training monitoring

# Import required libraries
import torch
import torch.distributed as dist
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import deepspeed
import accelerate
import ray
from ray import train, tune
import json
import os
import yaml
from typing import Dict, Any
import psutil
import GPUtil

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Check distributed setup
try:
    dist.init_process_group(backend='nccl')
    print(f"Process group initialized: {dist.get_rank()}/{dist.get_world_size()}")
    dist.destroy_process_group()
except:
    print("Single GPU/CPU setup detected")


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">DeepSpeed Training</h2>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">ZeRO Stages Overview</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">DeepSpeed ZeRO (Zero Redundancy Optimizer) provides different stages for memory optimization:</p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>ZeRO-0</strong>: Standard data parallelism</li>
<li style="margin:6px 0;"><strong>ZeRO-1</strong>: Optimizer state partitioning</li>
<li style="margin:6px 0;"><strong>ZeRO-2</strong>: Optimizer + gradient partitioning</li>
<li style="margin:6px 0;"><strong>ZeRO-3</strong>: Optimizer + gradient + parameter partitioning</li>
</ul>
</div>


In [ ]:
# DeepSpeed Configuration Files

# ZeRO-3 Configuration for maximum memory efficiency
ds_config_z3 = {
    "fp16": {
        "enabled": "auto",
        "auto_cast": False,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "bf16": {
        "enabled": "auto"
    },
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "offload_param": {
            "device": "cpu",
            "pin_memory": True
        },
        "overlap_comm": True,
        "contiguous_gradients": True,
        "sub_group_size": 1000000000000,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1000000000000,
        "stage3_max_reuse_distance": 1000000000000,
        "stage3_gather_16bit_weights_on_model_save": False
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "steps_per_print": 10,
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "wall_clock_breakdown": False
}

# ZeRO-2 Configuration for balanced memory and speed
ds_config_z2 = {
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 200000000,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 200000000,
        "contiguous_gradients": True
    }
}

# Save configurations
with open('ds_config_z3.json', 'w') as f:
    json.dump(ds_config_z3, f, indent=2)

with open('ds_config_z2.json', 'w') as f:
    json.dump(ds_config_z2, f, indent=2)
